# 02 — Feature Diagnostics

Decides which audio features go into the taste-profile vector used in 03 and as the bandit context in 05. Runs two independent checks:

1. **Multicollinearity** (§1) — Pearson correlation + VIF to detect *linear* redundancy
2. **Mutual Information** (§2) — captures *non-linear* dependence that correlation misses

§3 records the final 7-feature verdict consumed by every downstream notebook.

# 1. Multicollinearity


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the spotify cleaned data
df = pd.read_csv('outputs/spotify_cleaned.csv')

In [ ]:
## Dataset Overview
# Display first few rows
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

# Show data types
print("\n" + "="*60)
print("Data Types:")
print("="*60)
print(df.dtypes)

print("\n" + "="*60)
print("Column Information by Type:")
print("="*60)

# Categorize columns by type
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
datetime_cols = df.select_dtypes(include=['datetime64']).columns.tolist()
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Boolean columns ({len(bool_cols)}): {bool_cols}")
if datetime_cols:
    print(f"DateTime columns ({len(datetime_cols)}): {datetime_cols}")

In [ ]:
## 1) Correlation Matrix 
# Create correlation matrix (only for numeric columns)
correlation_matrix = df.corr(numeric_only=True)

print("\nCorrelation Matrix:")
print(correlation_matrix)

# Visualize the correlation matrix
plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Spotify Dataset - Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
## 2) VIF Analysis - Multicollinearity Testing for Feature Selection
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("VARIANCE INFLATION FACTOR (VIF) ANALYSIS")
print("="*80)
print("VIF measures how much the variance of a coefficient is inflated due to multicollinearity")
print("VIF > 10: High multicollinearity (drop feature)")
print("VIF 5-10: Moderate multicollinearity (consider dropping)")
print("VIF < 5: Acceptable level\n")

# Get numeric columns
numeric_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()

# Standardize features for VIF calculation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numeric_cols_all])
X_scaled_df = pd.DataFrame(X_scaled, columns=numeric_cols_all)

# Calculate VIF for all features
print("INITIAL VIF ANALYSIS (All Features)")
print("-" * 80)
vif_results = pd.DataFrame()
vif_results['Feature'] = numeric_cols_all
vif_results['VIF'] = [variance_inflation_factor(X_scaled, i) for i in range(X_scaled.shape[1])]
vif_results = vif_results.sort_values('VIF', ascending=False).reset_index(drop=True)

print(vif_results.to_string(index=False))

# Categorize features by VIF level
high_vif = vif_results[vif_results['VIF'] > 10]['Feature'].tolist()
moderate_vif = vif_results[(vif_results['VIF'] >= 5) & (vif_results['VIF'] <= 10)]['Feature'].tolist()
acceptable_vif = vif_results[vif_results['VIF'] < 5]['Feature'].tolist()

print(f"\n✓ Acceptable VIF (< 5): {len(acceptable_vif)} features")
print(f"   {acceptable_vif[:5]}{'...' if len(acceptable_vif) > 5 else ''}")

if moderate_vif:
    print(f"\n⚠️  Moderate multicollinearity (5-10): {len(moderate_vif)} features")
    print(f"   {moderate_vif}")

if high_vif:
    print(f"\n❌ High multicollinearity (> 10): {len(high_vif)} features")
    print(f"   {high_vif}")

# Iterative VIF reduction - drop highest VIF features one by one
print("\n" + "="*80)
print("ITERATIVE VIF REDUCTION")
print("="*80)
print("Dropping features with VIF > 10 iteratively until all remaining VIF < 10\n")

X_refined = X_scaled_df.copy()
dropped_features = []
iteration = 0

while True:
    iteration += 1
    vif_current = pd.DataFrame()
    vif_current['Feature'] = X_refined.columns
    vif_current['VIF'] = [variance_inflation_factor(X_refined.values, i) for i in range(X_refined.shape[1])]
    vif_current = vif_current.sort_values('VIF', ascending=False)
    
    max_vif = vif_current['VIF'].max()
    
    if max_vif > 10:
        feature_to_drop = vif_current.iloc[0]['Feature']
        drop_vif = vif_current.iloc[0]['VIF']
        print(f"Iteration {iteration}: Dropping '{feature_to_drop}' (VIF = {drop_vif:.2f})")
        X_refined = X_refined.drop(columns=[feature_to_drop])
        dropped_features.append(feature_to_drop)
    else:
        print(f"\n✓ Completed after {iteration-1} iterations")
        break

# Final VIF results
print("\n" + "="*80)
print("FINAL VIF ANALYSIS (Refined Features)")
print("="*80)
vif_final = pd.DataFrame()
vif_final['Feature'] = X_refined.columns
vif_final['VIF'] = [variance_inflation_factor(X_refined.values, i) for i in range(X_refined.shape[1])]
vif_final = vif_final.sort_values('VIF', ascending=False).reset_index(drop=True)

print(vif_final.to_string(index=False))

print("\n" + "="*80)
print("SUMMARY & RECOMMENDATIONS")
print("="*80)
print(f"Original features: {len(numeric_cols_all)}")
print(f"Refined features: {len(X_refined.columns)}")
print(f"Features dropped: {len(dropped_features)}")
if dropped_features:
    print(f"  → {dropped_features}")

print(f"\nMax VIF in original data: {vif_results['VIF'].max():.2f}")
print(f"Max VIF in refined data: {vif_final['VIF'].max():.2f}")
print(f"\nSuggested features for A/B testing (low multicollinearity):")
print(f"  {list(X_refined.columns)}")

# Create refined dataset for A/B testing
df_refined = df[list(X_refined.columns) + [col for col in df.columns if col not in numeric_cols_all]]
print(f"\nRefined dataset shape: {df_refined.shape}")
print(f"  Original: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"  Refined:  {df_refined.shape[0]} rows, {df_refined.shape[1]} columns")

In [ ]:
## 2.5) Interaction Terms Analysis on Refined Features
print("="*80)
print("INTERACTION TERMS ANALYSIS (Refined Features)")
print("="*80)
print("Testing which interaction terms are significant for A/B testing\n")

# Get refined features
refined_features = list(X_refined.columns)

# Show correlation matrix for refined features
correlation_refined = df_refined[refined_features].corr()

print("Correlation Matrix (Refined Features):")
print(correlation_refined.round(3))

# Find high correlations (excluding diagonal)
print("\n" + "-"*80)
print("High Correlations (|r| > 0.7) Among Refined Features:")
print("-"*80)

high_corr_pairs = []
for i in range(len(correlation_refined.columns)):
    for j in range(i+1, len(correlation_refined.columns)):
        corr_val = correlation_refined.iloc[i, j]
        if abs(corr_val) > 0.7:
            feat1 = correlation_refined.columns[i]
            feat2 = correlation_refined.columns[j]
            high_corr_pairs.append((feat1, feat2, corr_val))
            print(f"  {feat1} ↔ {feat2}: {corr_val:.3f}")

if not high_corr_pairs:
    print("  ✓ No high correlations (>0.7) found among refined features")

# Visualize correlation matrix for refined features
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_refined, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=0.5, cbar_kws={'label': 'Correlation'})
plt.title('Correlation Matrix - Refined Features (Low VIF)')
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("RECOMMENDED METRICS FOR A/B TESTING")
print("="*80)
print(f"Features (low multicollinearity, VIF < 10):")
for i, feat in enumerate(refined_features, 1):
    vif_val = vif_final[vif_final['Feature'] == feat]['VIF'].values[0]
    print(f"  {i}. {feat:20s} (VIF: {vif_val:6.2f})")

In [ ]:
## 3) Stricter VIF Reduction (VIF > 3)
print("="*80)
print("STRICTER VIF REDUCTION (VIF > 3)")
print("="*80)
print("Dropping features with VIF > 3 for even lower multicollinearity\n")

X_strict = X_refined.copy()
dropped_strict = []
iteration = 0

while True:
    iteration += 1
    vif_current = pd.DataFrame()
    vif_current['Feature'] = X_strict.columns
    vif_current['VIF'] = [variance_inflation_factor(X_strict.values, i) for i in range(X_strict.shape[1])]
    vif_current = vif_current.sort_values('VIF', ascending=False)
    
    max_vif = vif_current['VIF'].max()
    
    if max_vif > 3:
        feature_to_drop = vif_current.iloc[0]['Feature']
        drop_vif = vif_current.iloc[0]['VIF']
        print(f"Iteration {iteration}: Dropping '{feature_to_drop}' (VIF = {drop_vif:.2f})")
        X_strict = X_strict.drop(columns=[feature_to_drop])
        dropped_strict.append(feature_to_drop)
    else:
        print(f"\n✓ Completed after {iteration-1} iterations")
        break

# Final VIF results for strict
print("\n" + "="*80)
print("FINAL VIF ANALYSIS (Strict Features)")
print("="*80)
vif_strict = pd.DataFrame()
vif_strict['Feature'] = X_strict.columns
vif_strict['VIF'] = [variance_inflation_factor(X_strict.values, i) for i in range(X_strict.shape[1])]
vif_strict = vif_strict.sort_values('VIF', ascending=False).reset_index(drop=True)

print(vif_strict.to_string(index=False))

print("\n" + "="*80)
print("SUMMARY & RECOMMENDATIONS (Strict)")
print("="*80)
print(f"Previous refined features: {len(X_refined.columns)}")
print(f"Strict features: {len(X_strict.columns)}")
print(f"Additional features dropped: {len(dropped_strict)}")
if dropped_strict:
    print(f"  → {dropped_strict}")

print(f"\nMax VIF in strict data: {vif_strict['VIF'].max():.2f}")
print(f"\nSuggested features for A/B testing (very low multicollinearity):")
print(f"  {list(X_strict.columns)}")

# Create strict dataset
df_strict = df[list(X_strict.columns) + [col for col in df.columns if col not in numeric_cols_all]]
print(f"\nStrict dataset shape: {df_strict.shape}")
print(f"  Original: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"  Strict:  {df_strict.shape[0]} rows, {df_strict.shape[1]} columns")

In [ ]:
## 3.5) Correlation Analysis on Strict Features
print("="*80)
print("CORRELATION ANALYSIS (Strict Features)")
print("="*80)
print("Checking correlations among features with VIF ≤ 3\n")

# Get strict features
strict_features = list(X_strict.columns)

# Show correlation matrix for strict features
correlation_strict = df_strict[strict_features].corr()

print("Correlation Matrix (Strict Features):")
print(correlation_strict.round(3))

# Find high correlations (excluding diagonal)
print("\n" + "-"*80)
print("High Correlations (|r| > 0.7) Among Strict Features:")
print("-"*80)

high_corr_pairs_strict = []
for i in range(len(correlation_strict.columns)):
    for j in range(i+1, len(correlation_strict.columns)):
        corr_val = correlation_strict.iloc[i, j]
        if abs(corr_val) > 0.7:
            feat1 = correlation_strict.columns[i]
            feat2 = correlation_strict.columns[j]
            high_corr_pairs_strict.append((feat1, feat2, corr_val))
            print(f"  {feat1} ↔ {feat2}: {corr_val:.3f}")

if not high_corr_pairs_strict:
    print("  ✓ No high correlations (>0.7) found among strict features")

# Visualize correlation matrix for strict features
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_strict, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', square=True, linewidths=0.5, cbar_kws={'label': 'Correlation'})
plt.title('Correlation Matrix - Strict Features (VIF ≤ 3)')
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("FINAL RECOMMENDED METRICS FOR A/B TESTING")
print("="*80)
print(f"Features (very low multicollinearity, VIF ≤ 3):")
for i, feat in enumerate(strict_features, 1):
    vif_val = vif_strict[vif_strict['Feature'] == feat]['VIF'].values[0]
    print(f"  {i}. {feat:20s} (VIF: {vif_val:6.2f})")

# 2. Mutual-Information Independence


Complements the multicollinearity tests above (Pearson correlation + VIF, both linear) by checking for **non-linear** dependence via Mutual Information (MI).

- Pearson / VIF detect linear relationships only
- MI captures any statistical dependence, including non-linear
- MI = 0 ⇒ features independent; higher MI ⇒ more dependent
- MI > 0.3: high dependence (consider dropping one); 0.1–0.3: moderate

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('outputs/spotify_cleaned.csv')
print("Dataset shape:", df.shape)

In [ ]:
# Only the 8 audio features used for taste profiling in the bandit algorithm
TASTE_FEATURES = [
    'Danceability', 'Energy', 'Valence',
    'Acousticness', 'Instrumentalness', 'Speechiness',
    'Tempo', 'Loudness'
]

print(f"Features being tested ({len(TASTE_FEATURES)}): {TASTE_FEATURES}")

## Mutual Information Matrix

MI measures general statistical dependence. Unlike Pearson correlation, it captures **non-linear** relationships.

- MI = 0 ⇒ independent
- Higher MI ⇒ more shared information
- MI > 0.3: high dependence — consider dropping one
- MI 0.1–0.3: moderate

In [ ]:
# Compute pairwise MI matrix
n_features = len(TASTE_FEATURES)
mi_matrix = np.zeros((n_features, n_features))

print("Computing Mutual Information matrix...")

df_taste = df[TASTE_FEATURES]

for i, target_col in enumerate(TASTE_FEATURES):
    y = df_taste[target_col].values
    X = df_taste.drop(columns=[target_col]).values
    
    mi_values = mutual_info_regression(X, y, random_state=42)
    
    idx = 0
    for j in range(n_features):
        if j == i:
            mi_matrix[i, j] = np.nan
        else:
            mi_matrix[i, j] = mi_values[idx]
            idx += 1

# Symmetrize by averaging MI(i,j) and MI(j,i)
mi_symmetric = np.nanmean([mi_matrix, mi_matrix.T], axis=0)
np.fill_diagonal(mi_symmetric, np.nan)

mi_df = pd.DataFrame(mi_symmetric, index=TASTE_FEATURES, columns=TASTE_FEATURES)
print("Done.\n")
print("Mutual Information Matrix:")
print(mi_df.round(3))

In [ ]:
# Visualize MI matrix
plt.figure(figsize=(12, 10))
sns.heatmap(mi_df, annot=True, cmap='YlOrRd', fmt='.2f', square=True,
            linewidths=0.5, cbar_kws={'label': 'Mutual Information (nats)'},
            mask=np.eye(len(mi_df), dtype=bool))
plt.title('Mutual Information Matrix - Taste Profile Features\n(Higher = more dependent, 0 = independent)')
plt.tight_layout()
plt.show()

In [ ]:
# Flag high MI pairs
print("="*80)
print("HIGH MUTUAL INFORMATION PAIRS")
print("="*80)
print("Pairs with MI > 0.3 (notable non-linear dependence):\n")

high_mi_pairs = []
for i in range(n_features):
    for j in range(i+1, n_features):
        val = mi_df.iloc[i, j]
        if val > 0.3:
            high_mi_pairs.append((TASTE_FEATURES[i], TASTE_FEATURES[j], val))

high_mi_pairs.sort(key=lambda x: x[2], reverse=True)

if high_mi_pairs:
    for f1, f2, val in high_mi_pairs:
        print(f"  {f1:20s} <-> {f2:20s}  MI = {val:.3f}")
else:
    print("  No pairs with MI > 0.3 found.")

print("\nPairs with MI > 0.1 (moderate dependence):")
moderate_mi = [(f1, f2, v) for f1, f2, v in 
               [(TASTE_FEATURES[i], TASTE_FEATURES[j], mi_df.iloc[i, j])
                for i in range(n_features) for j in range(i+1, n_features)]
               if 0.1 < v <= 0.3]
moderate_mi.sort(key=lambda x: x[2], reverse=True)

if moderate_mi:
    for f1, f2, val in moderate_mi:
        print(f"  {f1:20s} <-> {f2:20s}  MI = {val:.3f}")
else:
    print("  No pairs with MI between 0.1 and 0.3 found.")

---
## Summary & Recommendation

In [ ]:
print("=" * 80)
print("SUMMARY")
print("=" * 80)

all_mi_pairs = []
for i in range(n_features):
    for j in range(i+1, n_features):
        all_mi_pairs.append((TASTE_FEATURES[i], TASTE_FEATURES[j], mi_df.iloc[i, j]))
all_mi_pairs.sort(key=lambda x: x[2], reverse=True)

print(f"\nTotal pairs tested: {len(all_mi_pairs)}")
print(f"  MI > 0.3 (high):       {sum(1 for _,_,v in all_mi_pairs if v > 0.3)}")
print(f"  MI 0.1-0.3 (moderate): {sum(1 for _,_,v in all_mi_pairs if 0.1 < v <= 0.3)}")
print(f"  MI <= 0.1 (low):       {sum(1 for _,_,v in all_mi_pairs if v <= 0.1)}")

print("\n" + "=" * 80)
print("FEATURES TO CONSIDER DROPPING")
print("=" * 80)

if high_mi_pairs:
    for f1, f2, val in high_mi_pairs:
        print(f"  Drop one of: {f1} / {f2} (MI={val:.3f})")
else:
    print("\nNo features flagged for removal - all features are sufficiently independent.")

print("\n" + "=" * 80)
print("DECISION CRITERIA")
print("=" * 80)
print("""
Mutual Information (MI) measures how much knowing one feature tells you
about another. MI = 0 means fully independent. Higher MI = more redundant.

Thresholds used:
  MI > 0.3  → High dependence: features share significant information.
              One of the pair should be dropped to avoid redundancy.
  MI 0.1-0.3 → Moderate: some shared info, but acceptable for modeling.
  MI < 0.1  → Low: features are effectively independent.

These thresholds are standard in feature selection literature.
""")

print("=" * 80)
print("RECOMMENDATION")
print("=" * 80)
print("""
High MI pairs found:
  1. Energy <-> Loudness       (MI = 0.643) — Strongest dependence
  2. Energy <-> Acousticness   (MI = 0.477)
  3. Acousticness <-> Loudness (MI = 0.362)
  4. Danceability <-> Valence  (MI = 0.307) — Borderline

Energy, Loudness, and Acousticness form a triangle of dependence.
All three measure aspects of song "intensity":
  - Energy: overall intensity
  - Loudness: volume level (correlates with intensity)
  - Acousticness: acoustic vs produced (inverse of intensity)

DROP: Loudness
  - It has the highest total MI with other features (0.643 + 0.362 = 1.005)
  - Energy already captures the "intensity" dimension
  - Acousticness already captures the "produced vs organic" dimension
  - Loudness adds little unique information beyond what these two provide

KEEP: Danceability and Valence
  - Their MI (0.307) is borderline and they capture distinct concepts:
    Danceability = rhythmic suitability, Valence = emotional positivity
  - Dropping either would lose meaningful taste information

FINAL FEATURE SET (7 features):
  Danceability, Energy, Valence, Acousticness,
  Instrumentalness, Speechiness, Tempo
""")

# 3. Feature-Selection Verdict

Combining the linear (§1) and non-linear (§2) diagnostics yields the following feature set for all downstream work (clustering in 03, taste profiles in 03, bandit context in 05):

| Feature          | Kept? | Reason |
|------------------|:-----:|--------|
| Danceability     | ✓     | Low VIF, low MI with others |
| Energy           | ✓     | Low VIF; carries the Loudness signal |
| Valence          | ✓     | Independent taste dimension |
| Acousticness     | ✓     | Independent taste dimension |
| Instrumentalness | ✓     | Independent taste dimension |
| Speechiness      | ✓     | Independent taste dimension |
| Tempo            | ✓     | Independent taste dimension (normalised downstream) |
| Loudness         | ✗     | MI ≈ 0.64 with Energy — redundant non-linear signal |
| Liveness         | ✗     | Weak taste signal, noisy |
| Duration_min     | ✗     | Not a taste dimension |
| EnergyLiveness   | ✗     | Engineered feature, redundant with Energy |

**Final feature set (7):** `Danceability, Energy, Valence, Acousticness, Instrumentalness, Speechiness, Tempo`.